<a href="https://colab.research.google.com/github/CainMacias/CainMacias/blob/main/California_housing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
import pandas as pd
from sklearn.datasets import fetch_california_housing

datos = fetch_california_housing(as_frame=True)
df = datos.frame
df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [24]:
# siguiente: separar X e y y hacer el train_test_split
df.shape

(20640, 9)

In [25]:
df.describe()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704,2.068558
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532,1.153956
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000,0.149990
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000,1.196000
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000,1.797000
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000,2.647250
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000,5.000010


In [26]:
df.isnull().sum()

,0
MedInc,0
HouseAge,0
AveRooms,0
AveBedrms,0
Population,0
AveOccup,0
Latitude,0
Longitude,0
MedHouseVal,0


In [27]:
y = df['MedHouseVal']
X = df.drop('MedHouseVal', axis = 1)
X.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [28]:
from sklearn.model_selection import train_test_split

train_X, val_X, train_y, val_y = train_test_split(X, y, random_state = 1)
print("Zonas para entrenar:", len(train_X))
print("Zonas para validar: ", len(val_X))

Zonas para entrenar: 15480
Zonas para validar:  5160


In [29]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

modelo_arbol = DecisionTreeRegressor(random_state = 1)
modelo_arbol.fit(train_X, train_y)

predicciones = modelo_arbol.predict(val_X)
mae_arbol = mean_absolute_error(val_y, predicciones)
print("Error promedio (MAE) del árbol:", round(mae_arbol, 1))
#0.5 de margen de error

Error promedio (MAE) del árbol: 0.5


In [30]:
print("Valor promedio de target:", round(y.mean(), 1))
print("Valor mínimo:", y.min(), "| Valor máximo:", y.max())

Valor promedio de target: 2.1
Valor mínimo: 0.14999 | Valor máximo: 5.00001


In [31]:
mae_entrenamiento = mean_absolute_error(train_y, modelo_arbol.predict(train_X))
mae_validacion = mean_absolute_error(val_y, predicciones)
print("Error en datos de entrenamiento", round(mae_entrenamiento, 1))
print("Error en datos de validación", round(mae_validacion, 1))

Error en datos de entrenamiento 0.0
Error en datos de validación 0.5


In [32]:
def get_mae(max_leaf_nodes, train_X, val_X, train_y, val_y):
  modelo = DecisionTreeRegressor(max_leaf_nodes = max_leaf_nodes, random_state = 1)
  modelo.fit(train_X, train_y)
  predicts = modelo.predict(val_X)
  mae = mean_absolute_error(val_y, predicts)
  return mae

tamaños = [5, 25, 50, 100, 250, 500, 1000, 2000]

scores = {}
for tamano in tamaños:
    mae = get_mae(tamano, train_X, val_X, train_y, val_y)
    scores[tamano] = mae           # para este tamaño, guarda este error
    print(tamano, round(mae, 3))

mejor_tamano = min(scores, key=scores.get)
print("Mejor tamaño:", mejor_tamano)

5 0.632
25 0.518
50 0.472
100 0.442
250 0.413
500 0.412
1000 0.422
2000 0.436
Mejor tamaño: 500


In [33]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

modelo_bosque = RandomForestRegressor(random_state=1)
modelo_bosque.fit(train_X, train_y)

mae_bosque = mean_absolute_error(val_y, modelo_bosque.predict(val_X))
print("Error promedio (MAE) del Random Forest:", round(mae_bosque, 3))

Error promedio (MAE) del Random Forest: 0.331


In [34]:
resultados = pd.DataFrame({
    'Modelo': ['Árbol sin límite', f'Árbol con {mejor_tamano} hojas', 'Random Forest'],
    'MAE en validación': [mae_arbol, scores[mejor_tamano], mae_bosque]
}).round(3)
resultados

,Modelo,MAE en validación
0,Árbol sin límite,0.464
1,Árbol con 500 hojas,0.412
2,Random Forest,0.331


In [35]:
importancias = pd.Series(modelo_bosque.feature_importances_, index=X.columns)
importancias.sort_values(ascending=False).round(3)

,0
MedInc,0.522
AveOccup,0.141
Latitude,0.092
Longitude,0.089
HouseAge,0.053
AveRooms,0.043
AveBedrms,0.031
Population,0.030
